In [ ]:

!pip install google-adk
!pip install google-cloud-aiplatform[agent_engines,adk]

In [ ]:
PROJECT_ID = "qwiklabs-gcp-01-171e30612222"
LOCATION = "us-central1"

In [ ]:
import subprocess
import os
import time

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging-{LOCATION}"

# 1. Enable Vertex AI API
print("1. Enabling Vertex AI API...")
subprocess.run(
    ["gcloud", "services", "enable", "aiplatform.googleapis.com", f"--project={PROJECT_ID}"],
    capture_output=True, text=True
)

# 2. Create staging bucket
print(f"2. Creating staging bucket: {STAGING_BUCKET}")
subprocess.run(["gsutil", "mb", "-l", LOCATION, STAGING_BUCKET], capture_output=True)

# 3. Get project number
result = subprocess.run(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
    capture_output=True, text=True
)
PROJECT_NUMBER = result.stdout.strip()
print(f"3. Project number: {PROJECT_NUMBER}")

# 4. Force-create the Reasoning Engine service agent
print("4. Ensuring Vertex AI service agent exists...")
subprocess.run(
    ["gcloud", "beta", "services", "identity", "create",
     "--service=aiplatform.googleapis.com", f"--project={PROJECT_ID}"],
    capture_output=True, text=True
)

# 5. Grant necessary roles to the RE service agent
SA_EMAIL = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
ROLES = ["roles/aiplatform.user", "roles/run.admin"]

for role in ROLES:
    print(f"   Granting {role} to {SA_EMAIL}...")
    subprocess.run(
        ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
         f"--member=serviceAccount:{SA_EMAIL}", f"--role={role}", "--quiet"],
        capture_output=True
    )

print("\nSetup complete. Please wait ~10 seconds before re-deploying the agent.")

1. Enabling Vertex AI API...
2. Creating staging bucket: gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1
3. Project number: 922001826607
4. Ensuring Vertex AI service agent exists...
   Granting roles/aiplatform.user to service-922001826607@gcp-sa-aiplatform-re.iam.gserviceaccount.com...
   Granting roles/run.admin to service-922001826607@gcp-sa-aiplatform-re.iam.gserviceaccount.com...

Setup complete. Please wait ~10 seconds before re-deploying the agent.


In [ ]:

import vertexai

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"Vertex AI initialized: {PROJECT_ID} / {LOCATION}")

Vertex AI initialized: qwiklabs-gcp-01-171e30612222 / us-central1


In [ ]:

from vertexai import agent_engines
import time

print("Checking for existing deployments to clean up...")
try:
    count = 0
    for existing in agent_engines.list():
        rn = getattr(existing, 'resource_name', None) or getattr(existing, 'name', None)
        dn = getattr(existing, 'display_name', 'unnamed')
        print(f"  Found: {dn} -> {rn}")

        # Try to delete — retry up to 3 times
        for attempt in range(3):
            try:
                agent_engines.delete(rn, force=True)
                print(f"  Deleted.")
                count += 1
                break
            except TypeError:
                # force= not supported — try without it
                try:
                    agent_engines.delete(rn)
                    print(f"  Deleted.")
                    count += 1
                    break
                except Exception as e2:
                    print(f"  Attempt {attempt+1}: {e2}")
                    time.sleep(5)
            except Exception as e:
                print(f"  Attempt {attempt+1}: {e}")
                time.sleep(5)

    if count == 0:
        print("  No existing deployments found. Clean slate.")
    else:
        print(f"\n  Deleted {count} deployment(s). Waiting 30 seconds...")
        time.sleep(30)
        print("  Ready to deploy.")
except Exception as e:
    print(f"  List failed: {e}")
    print("  Proceeding anyway — may be a clean slate.")



Checking for existing deployments to clean up...
  No existing deployments found. Clean slate.


In [ ]:
import time

def deploy_joke_teller():
    """
    Creates a minimal joke-telling agent and deploys it to Agent Engine.
    Everything is inside this function for clean serialization.
    """
    from google.adk.agents import LlmAgent
    from vertexai.preview.reasoning_engines import AdkApp
    from vertexai import agent_engines

    # Define the simplest possible agent — no tools, no callbacks
    # Appending a timestamp to the agent name to ensure uniqueness and avoid 409 conflicts.
    agent = LlmAgent(
        name=f"joke_teller_{int(time.time())}",
        model="gemini-2.0-flash",
        instruction="""You are a comedian. When the user asks for a joke,
        tell them a funny, clean joke. You can tell jokes about any topic.
        If the user specifies a topic, tailor the joke to that topic.
        Keep jokes family-friendly and light-hearted.""",
    )
    print(f"Agent created: {agent.name}")

    # Wrap in AdkApp
    app = AdkApp(agent=agent) # Removed deprecated enable_tracing parameter
    print("AdkApp created.\n")

    # Test locally first
    print("LOCAL TEST:")
    print("-" * 40)
    for event in app.stream_query(
        user_id="local-test",
        message="Tell me a joke about computers",
    ):
        print(event)
    print("-" * 40)
    print("Local test complete.\n")

    # Deploy
    print("Deploying to Agent Engine...")
    print("This may take 5-10 minutes...\n")

    remote = agent_engines.create(
        app,
        requirements=["google-cloud-aiplatform[agent_engines,adk]"],
    )

    print("=" * 60)
    print("DEPLOYMENT SUCCESSFUL!")
    print("=" * 60)
    print(f"Resource: {remote.resource_name}")
    return remote


remote_agent = deploy_joke_teller()

Agent created: joke_teller_1772825059
AdkApp created.

LOCAL TEST:
----------------------------------------


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.135.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requ

{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': 'Why did the computer get glasses?\n\nBecause it needed to improve its website!\n'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 17, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 17}], 'prompt_token_count': 91, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 91}], 'total_token_count': 108, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.04925217347986558, 'invocation_id': 'e-4c3ca351-b4c0-4fc7-ac8a-9ddcd70c6da6', 'author': 'joke_teller_1772825059', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'id': '0ea8fe86-695c-496c-bdf1-352e2786aa55', 'timestamp': 1772825059.69156}
----------------------------------------
Local test complete.

Deploying to Agent Engine...
This may take 5-10 minutes...



INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/922001826607/locations/us-central1/reasoningEngines/1070703323616641024/operations/2233315245105872896
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-01-171e30612222
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/922001826607/locations/us-central1/reasoningEngines/1070703323616641024
INFO:vertexai.agent_engines:To use thi

DEPLOYMENT SUCCESSFUL!
Resource: projects/922001826607/locations/us-central1/reasoningEngines/1070703323616641024


In [ ]:

print("Test 1: Computer joke")
print("=" * 60)

for event in remote_agent.stream_query(
    user_id="test-user",
    message="Tell me a joke about computers",
):
    print(event)

Test 1: Computer joke
{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': 'Why did the computer get glasses? \n\nBecause it needed to improve its website!\n'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 18, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 18}], 'prompt_token_count': 91, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 91}], 'total_token_count': 109, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.17463469505310059, 'invocation_id': 'e-656662f0-2ff2-44f7-a910-5f1ebf428752', 'author': 'joke_teller_1772825059', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'id': 'e86c8786-287a-4de8-ba52-4b706ad997ee', 'timestamp': 1772825573.311827}


In [ ]:
print("Test 2: Science joke")
print("=" * 60)

for event in remote_agent.stream_query(
    user_id="test-user",
    message="Tell me a joke about science",
):
    print(event)

Test 2: Science joke
{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': 'Why did the atom cross the road?\n\nBecause it heard there was plenty of potential!\n'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 19, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 19}], 'prompt_token_count': 91, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 91}], 'total_token_count': 110, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.1383477386675383, 'invocation_id': 'e-536a0f36-d6f9-4b6a-a2ef-f33e3a9a10a8', 'author': 'joke_teller_1772825059', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'id': 'a325393c-3e54-43eb-bff9-b191758add0e', 'timestamp': 1772825575.722068}


In [ ]:
print("Test 3: Surprise me")
print("=" * 60)

for event in remote_agent.stream_query(
    user_id="test-user",
    message="Surprise me with your funniest joke",
):
    print(event)

Test 3: Surprise me
{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': "Why don't scientists trust atoms?\n\nBecause they make up everything!\n"}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 16, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 16}], 'prompt_token_count': 92, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 92}], 'total_token_count': 108, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.045200612396001816, 'invocation_id': 'e-5a69a04b-8d04-4dc8-998c-b46e0f167460', 'author': 'joke_teller_1772825059', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'id': 'a7eb020e-e1b2-4dc1-b616-42df859f0b60', 'timestamp': 1772825577.604745}


In [ ]:

print("""
╔══════════════════════════════════════════════════════════════════╗
║           CHALLENGE 5 — REQUIREMENTS VALIDATION                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  REQUIREMENT 1: Create an agent using the ADK          ✅        ║
║  → LlmAgent "joke_teller" with gemini-2.0-flash                 ║
║  → Defined inside deploy_joke_teller() for clean serialization   ║
║                                                                  ║
║  REQUIREMENT 2: Deploy the agent to Agent Engine       ✅        ║
║  → AdkApp wrapper + agent_engines.create()                       ║
║  → All previous deployments cleaned before creating              ║
║  → IAM + bucket + API all configured in Cell 3                   ║
║                                                                  ║
║  REQUIREMENT 3: Test the agent                         ✅        ║
║  → Local test inside deploy function (Cell 6)                    ║
║  → Remote tests: Cells 7, 8, 9                                  ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

try:
    print(f"Deployed: {remote_agent.resource_name}")
    print("Status: OPERATIONAL")
except:
    print("WARNING: remote_agent not found — rerun Cell 6")


╔══════════════════════════════════════════════════════════════════╗
║           CHALLENGE 5 — REQUIREMENTS VALIDATION                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  REQUIREMENT 1: Create an agent using the ADK          ✅        ║
║  → LlmAgent "joke_teller" with gemini-2.0-flash                 ║
║  → Defined inside deploy_joke_teller() for clean serialization   ║
║                                                                  ║
║  REQUIREMENT 2: Deploy the agent to Agent Engine       ✅        ║
║  → AdkApp wrapper + agent_engines.create()                       ║
║  → All previous deployments cleaned before creating              ║
║  → IAM + bucket + API all configured in Cell 3                   ║
║                                                                  ║
║  REQUIREMENT 3: Test the agent                         ✅        ║
║  → Local test inside deploy functio

In [ ]:
# Uncomment to delete and avoid charges:
# from vertexai import agent_engines
# agent_engines.delete(remote_agent.resource_name)
# print("Deleted.")